# Final Project, Part 2

The second component of your project will be to design your "observatory." Note that this is pseudo realtime. It does not need to automatically update its data; a refresh button, or requiring a page reload, is sufficient. This is worth 40 points out of 100.

This could consist of drawings, prototypes of web pages or notebooks, and the code necessary to build the different components. This component is scaffolding, not the final product. It should be thought of as the "viz for peers" stage of the process. You can submit drawings (including photos of drawings), jamboards, notebooks (which you are allowed to clean up), and stages of prototypes.

You should walk me through the process of how you built your observatory, and it does not need to be a linear, successes-only process.

### Project Process & Observations
We started the project by opening the 2024 Stack Overflow Developer Survey with pandas in a Jupyter notebook. The dataset was big and contained numerous columns, so we initially meticulously picked the most useful ones — such as the developer's country, experience, job role, salary, languages used, and how they learned coding.

After we chose the columns, we cleaned the data. There were columns such as LanguageHaveWorkedWith and LearnCode which contained more than one value in a single cell separated by semicolons. We needed to split them using .str.split(';') and then spread the values into new rows using .explode().

One issue was with the YearsCode and YearsCodePro columns, which contained text such as "Less than 1 year" or "More than 50 years." We replaced these with numbers (0 and 51 respectively) so that we could include them in calculations and plots.

For compensation (CompTotal), the figures ranged very widely and contained outliers that would skew the averages. We chose to retain only figures between the 5th and 95th percentile to make the graphs more readable and more insightful.

We used both of the prominent visualization libraries — Altair and Plotly. Altair was best suited for bar and pie charts with interactive and responsive features, while Plotly was more appropriate for dynamic charts such as histograms and bubble plots.

The design process was not always linear. For instance, we initially had a simple bar chart of use of AI tools, but then realized that it didn't tell the entire story. We improved it by representing the same tasks categorized by "currently using," "interested," and "not interested," so trends could be more easily discerned.

Along the way, we worked hard to build interactive charts — through dropdowns and click filters — to allow individuals to find the data for themselves. In the end, our observatory serves to uncover a clear story of how developers learn, work, and use tools like AI around the globe.

## Code:





In [ ]:
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import altair as alt
import plotly.graph_objects as go

In [ ]:
##from google.colab import files
##uploaded = files.upload()

In [ ]:
# Load the dataset
df = pd.read_csv("survey_results_public.csv")

### ------------------------ 1. Learning Methods by Experience ------------------------

In [ ]:
# Clean and preprocess learning experience data
df_learn = df[['YearsCode', 'LearnCode']].dropna()
df_learn['YearsCode'] = df_learn['YearsCode'].replace({'Less than 1 year': 0, 'More than 50 years': 51}).astype(float)
df_learn['LearnList'] = df_learn['LearnCode'].str.split(';')
df_learn = df_learn.explode('LearnList')
learn_exp = df_learn.groupby('LearnList').agg(avg_exp=('YearsCode', 'mean'), count=('YearsCode', 'count')).reset_index()
learn_exp = learn_exp[learn_exp['count'] > 50]

In [ ]:
# Normalize experience for color mapping
learn_exp['norm_exp'] = (learn_exp['avg_exp'] - learn_exp['avg_exp'].min()) / (learn_exp['avg_exp'].max() - learn_exp['avg_exp'].min())

In [ ]:
# Plot interactive pictogram-style bubble plot
fig = px.scatter(
    learn_exp,
    x="LearnList",
    y="avg_exp",
    size="count",
    color="norm_exp",
    color_continuous_scale="Inferno",
    size_max=60,
    hover_name="LearnList",
    hover_data=["count", "avg_exp"]
)

# Add line plot connecting the bubbles
fig.add_trace(go.Scatter(
    x=learn_exp["LearnList"],
    y=learn_exp["avg_exp"],
    mode='lines+markers',
    line=dict(color='rgba(0, 0, 0, 0.3)', width=2),
    marker=dict(size=5, color='black'),
    showlegend=False
))

# Update layout for readability
fig.update_layout(
    title="Learning Methods vs Experience",
    xaxis_title="Learning Resource",
    yaxis_title="Average Experience (Years)",
    xaxis_tickangle=-45,
    margin=dict(t=60, l=50, r=50, b=200),
    height=600,
    template="plotly_white"
)

fig.show()

### Plot 1 Summary: Learning Methods vs Experience
This graph investigates how coding education interacts with years of experience. We examined the LearnCode column, which contains the various methods by which each respondent learned — books, online courses, bootcamps, school, or learning on the job. Because most individuals provided more than one method, we separated these into separate rows so that we could examine each learning type individually.

We then calculated the average years of experience for each method using the YearsCode column. We also excluded rare responses by only looking at methods that had been reported by 50 or more users.

In the final chart, each bubble represents a learning method. The Y-position is the average years of experience for developers who learned through that method. The size of the bubble represents how many people used it, and the color represents how experienced those developers are (darker = more experience). A dotted line connects the bubbles to make the trend visually easier to spot.

What we can see from this visualization is that the developers who learned by books or on-the-job training have higher years of experience. On the other hand, bootcamps or online training are more popular among newer developers.

This graph assists us in comprehending which learning pathways are more typical for veteran developers and which ones beginners favor.

### ------------------------ 2. Top Programming Languages & Top 15 Countries ------------------------

In [ ]:
alt.data_transformers.enable('default', max_rows=None)

lang_df = df[['Country', 'LanguageHaveWorkedWith']].dropna()

lang_df = lang_df.assign(Language=lang_df['LanguageHaveWorkedWith'].str.split(';')).explode('Language')

lang_df['Language'] = lang_df['Language'].str.strip()

lang_counts = lang_df['Language'].value_counts().head(15).reset_index()
lang_counts.columns = ['Language', 'Count']

click = alt.selection_point(fields=['Language'])

left_chart = alt.Chart(lang_counts).mark_bar().encode(
    x='Count:Q',  # number of users
    y=alt.Y('Language:N', sort='-x'),  
    tooltip=['Language:N', 'Count:Q'],  
    color=alt.condition(click, alt.value('#1f77b4'), alt.value('#d3d3d3'))  
).add_params(click).properties(
    width=300,
    height=400,
    title='Top Programming Languages Globally'
)

In [ ]:
right_data = lang_df.groupby(['Language', 'Country']).size().reset_index(name='Count')

pie_chart = alt.Chart(right_data).transform_filter(
    click  
).transform_window(
    rank='rank(Count)', 
    sort=[alt.SortField('Count', order='descending')],
    groupby=['Language']
).transform_filter(
    alt.datum.rank <= 10
).encode(
    theta='Count:Q',
    color=alt.Color('Country:N', legend=alt.Legend(title="Country")),
    tooltip=['Country:N', 'Count:Q']
).mark_arc(innerRadius=50, outerRadius=120).properties(
    width=350,
    height=400,
    title='Top 10 Country Share (Donut)'
)

chart = left_chart | pie_chart
chart.spacing = 50

chart

### Plot 2 Summary: Country Trends and Programming Language Use
This interactive graph assists us in comprehending what the most popular programming languages are on the planet, and how they differ nation by nation. We utilized the LanguageHaveWorkedWith column, which contains all of the programming languages with which a particular respondent has worked. Because some developers included numerous languages, we separated the data so that each language is in its own row.

We counted the mentions of each language to find the top 15 most popular programming languages in the world. These are popular languages like JavaScript, Python, SQL, and HTML/CSS. We graphed this in a horizontal bar chart on the left side of the screen.

To make the visualization more useful, we included interactivity. If you click on any of the languages in the bar chart, the donut chart to the right will update to reflect the top 10 countries where the language is most used. For instance, if you click on Python, the donut chart can indicate that the U.S., India, and Germany have the highest number of Python developers.

This overlay of two charts allows us to examine both worldwide popularity and regional trends simultaneously. It's helpful for seeing which languages are popular everywhere and where particular languages have a strong following. This information is helpful for global teams, developers deciding which language to learn, or anyone who cares about tech trends around the world.

### ------------------------ 3. Distribution of Professional Coding Experience ------------------------

In [ ]:
# Clean and filter the 'YearsCodePro' column
df = df[['YearsCodePro']].dropna()
df['YearsCodePro'] = df['YearsCodePro'].replace({
    'Less than 1 year': 0,
    'More than 50 years': 51
})
df['YearsCodePro'] = pd.to_numeric(df['YearsCodePro'], errors='coerce')
df = df.dropna()

# Create interactive histogram + KDE using Plotly
fig = px.histogram(
    df,
    x='YearsCodePro',
    nbins=50,
    marginal="rug",  # adds individual value markers
    opacity=0.75,
    color_discrete_sequence=['#1f77b4'],
    title="Distribution of Professional Coding Experience (Interactive)"
)

fig.update_layout(
    xaxis_title="Years of Professional Coding",
    yaxis_title="Count",
    bargap=0.05,
    hovermode="x unified"
)

fig.show()

### Plot 3 Summary: Distribution of Professional Coding Experience
This visualization shows the distribution of professional coding experience among developers, mainly focusing on column “YearsCodePro” from Stack Overflow Developer Survey 2024. In order to maintain plotting consistency, data types are originally strings (e.g., "Less than 1 year", "More than 50 years"), which we convert into numeric format using replacement and pd.to_numeric(). In the plotting process, we first introduce removing NA values by .dropna() and also turn "Less than 1 year" and "More than 50 years" to 0 and 51. As for visualization, we utilized Plotly to create an interactive histogram with 50 bins to demonstrate experience distribution. We also create interactivity in this plot that allows users to hover and examine the count of survey participants with a different range of experience. It is an intuitive way to observe the overall experience among professional developers. With this plot, we found out most developers have 0 to 10 years of experience and very few have more than 30 years. It helps us understand the general experience level in the developer community.

### ------------------------ 4. AI Task Adoption (Current vs Interested vs Not Interested) ------------------------

In [ ]:
df = pd.read_csv('survey_results_public.csv', low_memory=False, nrows=10000)
ai_cols = ['AIToolCurrently Using', 'AIToolInterested in Using', 'AIToolNot interested in Using']
ai_long = pd.DataFrame(columns=['Task', 'Status'])

for col, label in zip(ai_cols, ['Currently Using', 'Interested', 'Not Interested']):
    temp = df[[col]].dropna().copy()
    temp['Status'] = label
    temp['Task'] = temp[col].str.split(';')
    temp = temp.explode('Task')[['Task', 'Status']]
    ai_long = pd.concat([ai_long, temp], axis=0)

ai_long = ai_long.groupby(['Task', 'Status']).size().reset_index(name='Count')
top_tasks = ai_long.groupby('Task')['Count'].sum().nlargest(12).index
ai_long = ai_long[ai_long['Task'].isin(top_tasks)]

In [ ]:
status_param = alt.param(
    name='StatusSelector',
    bind=alt.binding_select(options=['All', 'Currently Using', 'Interested', 'Not Interested'], name='Filter by Status:'),
    value='All'
)

chart = alt.Chart(ai_long).mark_bar().encode(
    y=alt.Y('Task:N', sort='-x'),
    x=alt.X('Count:Q'),
    color='Status:N',
    tooltip=['Task', 'Status', 'Count']
).add_params(
    status_param
).transform_filter(
    (alt.datum.Status == status_param) | (status_param == 'All')
).properties(
    title='AI Task Adoption: Current vs Interested vs Not Interested',
    width=700
)

chart

### Plot 4 Summary: AI Task Adoption: Current vs Interested vs Not Interested
This visualization explores how developers use and adopt AI tools across various software development tasks using data from the 2024 Stack Overflow Developer Survey. Currently Using, Interested in Using, and Not Interested in Using are the crucial columns that were extracted. Each column consists of semicolon-separated values, so we first transformed the data into long format using .explode() after splitting the responses. We also used .dropna() to exclude empty or incomplete entries and aggregated response counts using .groupby(). To also ensure readability, we restricted our chart to the top 12 AI tasks by total counts. Each task's adoption status breakdown of the three categories (Currently Using, Interested, and Not Interested) is reflected in the processed data.
 
For plotting, we employed Altair to plot a stacked horizontal bar chart with filtering capability. A dropdown selector allows filtering tasks by adoption status. This facilitates users to explore interest patterns and current usage without interference. The color bars offers a clear indication of how AI adoption varies across tasks like coding, testing, and project planning. Light blue represents "Currently Using" status, dark blue for "Interested", and red for "Not Interested". The visualization also shows that most developers already use AI to do things like code and look up answers, and interest is growing in doing things like documentation and project planning.

### ------------------------ 5. Average Compensation by Job Position Type ------------------------

In [ ]:
alt.data_transformers.disable_max_rows()

df = pd.read_csv("survey_results_public.csv", low_memory=False)

df_salary_clean = df[['Country', 'DevType', 'CompTotal']].dropna()
df_salary_clean['CompTotal'] = df_salary_clean['CompTotal'].astype(int)
df_salary_clean['DevType'] = df_salary_clean['DevType'].apply(lambda x: 'Other' if str(x).startswith('Other (please specify)') else x.strip())

low, high = df_salary_clean['CompTotal'].quantile([0.05, 0.95])
df_salary_filtered = df_salary_clean[(df_salary_clean['CompTotal'] >= low) & (df_salary_clean['CompTotal'] <= high)]


dropdown = alt.binding_select(
    options=[
        'United States of America',
        'Germany',
        'India',
        'United Kingdom of Great Britain and Northern Ireland',
        'Ukraine',
        'France',
        'Canada',
        'Poland',
        'Netherlands',
        'Brazil'
    ],
    name='Country: '
)

country_param = alt.param(
    value='United States of America',
    bind=dropdown,
    name='selected_country'
)

salary_bar_chart = alt.Chart(df_salary_filtered).mark_bar().encode(
    x=alt.X('DevType:N', sort='-y', title='Job Position Type'),
    y=alt.Y('CompTotal:Q', aggregate='mean', title='Average Compensation (local currency)'),
    color=alt.Color('DevType:N', legend=alt.Legend(title="Developer Type"), scale=alt.Scale(scheme='tableau20')),
    tooltip=['DevType:N',alt.Tooltip('CompTotal:Q', aggregate='mean', title='Avg Compensation')]
).add_params(
    country_param
).transform_filter(
    alt.datum.Country == country_param
).properties(
    title='Average Compensation by Job Position Type (Select Top10 Countries)',
    width=800,
    height=400
).configure_axisX(labelAngle=-45)

salary_bar_chart

### Plot 5 Summary: Average Compensation by Job Position Type
This visualization tries to demonstrate average compensation for different developer job types in top 10 dataset countries that are measured in their own local currencies. Job types include various developer positions. Our datasource came from Stack Overflow Developer Survey 2024 which focuses on several columns among comprehensive datasets such as “Country”, “DevType” and “CompTotal”. Data Types are String for job titles and countries while integer for total compensation. In order to create this visualization, we implemented several steps and first was dataset exploration. We removed null or missing values using .dropna(); also, grouped hand typing responses (e.g., "Other (please specify)") into a single "Other" category by using lambda function. Next, we tried to filter out extreme outliers outside the 5th and 95th percentile since we are focusing on average compensation and afraid of skewing the mean. For plotting tools, this plot implemented Altair that also added a dropdown menu to filter by the top 10 countries with tooltips for more user friendly. The dropdown interactivity allows the bar chart to update its data when the user makes a selection on a specific country.

## Prose:

* Write-up for explaining the process of how you built your observatory.


### Building the Developer Observatory: Our Process
Our goal for this project was to create an interactive “developer observatory” using the 2024 Stack Overflow Developer Survey. The observatory would allow users to explore patterns in how developers learn, work, and use new technologies like AI. Here's how we approached the process step by step:

### 1. Understanding the Dataset

We began by carefully examining the dataset structure. It contained thousands of responses and dozens of columns, many of which were multi-valued (e.g., multiple languages or tools listed in one field). To make sense of this, we first:
Explored all columns using df.info() and df.head()
Identified key variables of interest, including:
LanguageHaveWorkedWith (languages used)
Country
YearsCode and YearsCodePro
LearnCode (how the individual learned to code)
CompTotal (total compensation)
DevType (job type)
AI-related columns

### 2. Data Cleaning & Transformation

This step involved preparing the data for visualization:
Converted text-based ranges like "Less than 1 year" and "More than 50 years" into numeric values (0 and 51)
Split semicolon-separated fields (e.g., LanguageHaveWorkedWith, LearnCode) using .str.split(';') and .explode() to analyze individual items
Used.dropna() to drop rows containing missing values in critical columns
Applied percentile-based filtering (5th–95th) on CompTotal to reduce the effect of outliers on compensation averages
Grouped and aggregated the data by using .groupby() to calculate counts and means

### 3. Choosing Visualization Tools

We experimented with different Python libraries to determine which would be most effective:
Plotly was chosen for dynamic and interactive visuals like bubble charts and histograms
Altair was used for building clean, interactive bar charts, donut charts, and dropdown-based selectors
Both tools were chosen to support user interactivity, enabling viewers to explore the data beyond static graphs

### 4. Designing the Visualizations

We created five main plots, each showing a different aspect of the developer population:
Learning Methods vs Experience: a bubble chart showing average experience for each learning method
Programming Language Trends by Country:a bar + donut chart combo showing global and regional language popularity
Years of Coding Experience: a histogram of professional experience levels
AI Tool Adoption: a stacked bar chart of current vs interested vs not interested users by task
Compensation by Developer Type: an interactive bar chart filtered by country
Each visualization was made to be engaging and informative, balancing detail with readability.

### 5. Developing the Web Interface

Once our charts were ready, we built an interactive dashboard using Streamlit, which:
Displays each visualization in its own tab
Offers dropdowns or clickable elements for user-driven exploration
Includes markdown summaries to explain the insights in plain language
We then deployed the dashboard to Hugging Face Spaces for others to easily interact with.

### 6. Reflection and Iteration

The observatory was built iteratively. For example: We revised the AI chart from a basic bar chart to a more informative color-coded stacked layout We adjusted data filters for better performance on large datasets We refined labels, titles, and tooltips for clarity Along the way, we worked together to split up tasks and check each other's code and summaries.

## Plot Summary

Summarize the characteristics of the dataset in words: what does it represent, what are the fields/columns/rows, what data types are they, etc.

### Dataset Features

The data for the project is from the **2024 Stack Overflow Developer Survey**, a collection of responses from paid and volunteer software developers from across the globe. It is self-reported data collected through a well-structured survey questionnaire, with a single **row for each distinct respondent** and a single **column for each survey question provided or metadata field**.
It contains over **250 columns** and **tens of thousands of rows**, spanning numerous facets of developer life and work. They include:

- **Demographic data** (e.g., `Country`, `Age`, `Gender`, `EdLevel`)
- **Work and role details** (e.g., `DevType`, `OrgSize`, `WorkExp`)
- **Technical experience** (e.g., `YearsCode`, `YearsCodePro`)
- **Technologies used** (i.e., `LanguageHaveWorkedWith`, `DatabaseHaveWorkedWith`, `PlatformHaveWorkedWith`)
- **Learning background** (e.g., `LearnCode`)
- **Compensation data** (`CompTotal`, `Currency`)
- **Attitudes towards and usage of AI tools** (`AIToolCurrentlyUsing`, `AIToolInterested in Using`, `AIToolNot interested in Using`)

The data types vary by column:
- **Numerical fields** such as `CompTotal` are typically floats or integers
- **Categorical columns** like `Country` or `DevType` are stored as strings
- **Ordinal fields** like `YearsCodePro` and `YearsCode` contain numeric ranges and text categories (e.g., "Less than 1 year"), which need to be converted for analysis
- **Multi-select fields** such as `LanguageHaveWorkedWith` and `LearnCode` use semicolon-delimited strings to encode multiple responses in one cell

This combination of semi-structured and structured data required a number of preprocessing steps:

- We used `.dropna()` to handle missing values in critical columns
- We applied `.str.split(';')` and `.explode()` for flatted multi-select fields aggregations - Normalized ordinal text fields by translating qualitative values into numeric representations (e.g., "Less than 1 year" → 0) - Compensation figures were passed through the 5th–95th percentile range to reduce the effect of extreme outliers In total, the dataset paints a **complete and nuanced picture of worldwide developer trends**, apt both for statistical analysis and interactive data visualization. It records quantitative measures (such as salary or experience) and qualitative observations (such as tool usage and learning habits), making it perfect to create an observatory to find out how developers learn, work, and evolve with new technologies.